In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets

In [ ]:
!pip uninstall -y torchao
!pip install -q torchao==0.16.0

In [1]:
import torch
import torchao
import peft

print("Torch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("PEFT:", peft.__version__)

Torch: 2.10.0+cu128
TorchAO: 0.16.0
PEFT: 0.19.1


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [ ]:
from huggingface_hub import login

login()

In [8]:
!git clone https://github.com/MR-just01/Llama3.2-Reasoning

Cloning into 'Llama3.2-Reasoning'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 122 (delta 67), reused 31 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 21.55 MiB | 12.38 MiB/s, done.
Resolving deltas: 100% (67/67), done.


In [9]:
import pandas as pd

DATA_PATH = "/kaggle/working/Llama3.2-Reasoning/data/processed/reasoning_dataset (1).csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (30193, 6)
['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type']


In [11]:
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [12]:
def format_prompt(row):

    messages = [
        {
            "role": "user",
            "content": f"{row['instruction']}\n\n{row['input']}"
        },
        {
            "role": "assistant",
            "content": (
                f"Reasoning:\n{row['reasoning']}\n\n"
                f"Final Answer:\n{row['answer']}"
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

df["text"] = df.apply(format_prompt, axis=1)

In [13]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

Training samples: 27173
Validation samples: 3020


In [14]:
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER = "MR023/Llama3.2-Reasoning"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER
)

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
question = "If five machines make five toys in five minutes, how long would it take 100 machines to make 100 toys?"

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working/Llama3.2-Reasoning/data"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=False
)

In [ ]:
train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

In [ ]:
for i in range(start_idx, len(val_df)):

    row = val_df.iloc[i]

    prompt = f"{row['instruction']}\n\n{row['input']}"

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generation_time = time.time() - start_time

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    results.append({
        "index": row.name,
        "instruction": row["instruction"],
        "input": row["input"],
        "expected_answer": str(row["answer"]),
        "model_response": response,
        "generation_time": generation_time,
        "dataset": row["dataset"],
        "task_type": row["task_type"]
    })

    # Save every 100 examples
    if (i + 1) % 100 == 0:
        results_df = pd.DataFrame(results)
        results_df.to_csv(results_path, index=False)

        print(
            f"Processed {i + 1}/{len(val_df)} | "
            f"Saved: {results_path}"
        )

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [47]:
#loading the datset using the git
import pandas as pd

results_path = "/kaggle/input/datasets/mayarawat/validation2/validation_results (2).csv"

results_df = pd.read_csv(results_path)

print("Shape:", results_df.shape)

Shape: (3000, 8)


In [50]:
eval_df = results_df.copy()
eval_df["has_final_answer"] = (
    eval_df["model_response"]
    .str.contains("Final Answer:", case=False, na=False)
)

eval_df["response_length"] = (
    eval_df["model_response"]
    .fillna("")
    .str.len()
)

print("Total responses:", len(eval_df))
print(
    "With Final Answer:",
    eval_df["has_final_answer"].sum()
)
print(
    "Without Final Answer:",
    (~eval_df["has_final_answer"]).sum()
)

Total responses: 3000
With Final Answer: 2720
Without Final Answer: 280


In [51]:
#inspecting the 280 failure 
failed_df = eval_df[
    ~eval_df["has_final_answer"]
].copy()

print("Failed/incomplete responses:", len(failed_df))

display(
    failed_df[
        [
            "index",
            "expected_answer",
            "model_response"
        ]
    ].head(10)
)

Failed/incomplete responses: 280


,index,expected_answer,model_response
20,9538,6,"Reasoning:\n8k8\n+ k88\n--------\n1,6t6\n8k8 +..."
29,22954,8,Reasoning:\n230/18 = 12.78\n230/10 = 23\n12.78...
33,24876,20,Reasoning:\n50^8 × 8^3 × 11^3 = 50^8 × 8^3 × 1...
35,9489,53/99,Reasoning:\nExplanation:\n0.535353... = 535353...
37,2388,$7.85,Reasoning:\nLet x be the hourly charge to rent...
45,17935,-4,Reasoning:\n#p# = ap^3+ bp – 1\n#p# = ap^3+ bp...
62,13722,"-1/3 < x and x< 0, x > 2/5",Reasoning:\n15x - 2/x > 1\n15x - 1 > 2/x\n15x ...
68,2208,21,Reasoning:\nExplanation:\nLet us consider the ...
69,25152,50,Reasoning:\nExplanation:\nLet the rate of inte...
72,2767,"$192,500",Reasoning:\nLet's break down the problem:\n30%...


In [52]:
def repetition_ratio(text):
    if pd.isna(text) or not str(text).strip():
        return 0.0

    lines = [
        line.strip()
        for line in str(text).splitlines()
        if line.strip()
    ]

    if len(lines) <= 1:
        return 0.0

    unique_lines = len(set(lines))

    return 1 - (unique_lines / len(lines))



In [54]:
eval_df["repetition_ratio"] = (
    eval_df["model_response"]
    .apply(repetition_ratio)
)

In [55]:
eval_df["repetitive"] = (
    eval_df["repetition_ratio"] >= 0.5
)

print(
    "Highly repetitive responses:",
    eval_df["repetitive"].sum()
)

Highly repetitive responses: 182


In [56]:
eval_df["too_short"] = (
    eval_df["response_length"] < 20
)

print(
    "Very short responses:",
    eval_df["too_short"].sum()
)

Very short responses: 0


In [57]:
print("===== LLAMA RESPONSE SUMMARY =====")

print("Total responses:", len(eval_df))

print(
    "With Final Answer:",
    eval_df["has_final_answer"].sum()
)

print(
    "Without Final Answer:",
    (~eval_df["has_final_answer"]).sum()
)

print(
    "Highly repetitive:",
    eval_df["repetitive"].sum()
)

print(
    "Very short:",
    eval_df["too_short"].sum()
)

===== LLAMA RESPONSE SUMMARY =====
Total responses: 3000
With Final Answer: 2720
Without Final Answer: 280
Highly repetitive: 182
Very short: 0


In [62]:
def extract_final_answer(text):
    if pd.isna(text):
        return None

    text = str(text)

    marker = "Final Answer:"

    if marker.lower() not in text.lower():
        return None

    # Find marker without worrying about capitalization
    pos = text.lower().rfind(marker.lower())

    answer = text[pos + len(marker):].strip()

    # Keep only the first non-empty line
    lines = [line.strip() for line in answer.splitlines() if line.strip()]

    if not lines:
        return None

    return lines[0]


eval_df["extracted_answer"] = (
    eval_df["model_response"]
    .apply(extract_final_answer)
)

display(
    eval_df[
        [
            "expected_answer",
            "extracted_answer",
            "model_response"
        ]
    ].head(10)
)

,expected_answer,extracted_answer,model_response
0,34,34,Reasoning:\nTwice as many hats as fire chief S...
1,228 cm2,228 cm2,Reasoning:\nArea of trapezium = 1/2 * (sum of ...
2,43,47,Reasoning:\nThe number must be divisible by 21...
3,20,20,Reasoning:\nLet x be the number of birds on th...
4,6,6,Reasoning:\nFirst find the total distance Tod ...
5,B)1/8,C)1/9,Reasoning:\nThe probability that the first let...
6,1/7,1/7,Reasoning:\nLet x be the fraction of the dista...
7,4800,"4,800","Reasoning:\nEach month, Diego saves $5,000 - $..."
8,10years,10years,Reasoning:\nExplanation:\nTotal age of 15 stud...
9,65,65,Reasoning:\nThe total cost of sweatshirts is 3...


In [63]:
print(
    "Answers successfully extracted:",
    eval_df["extracted_answer"].notna().sum()
)

print(
    "Answers not extracted:",
    eval_df["extracted_answer"].isna().sum()
)

Answers successfully extracted: 2720
Answers not extracted: 280


In [66]:
#normalization of the answers
import re

def normalize_answer(answer):
    if pd.isna(answer):
        return None

    answer = str(answer).strip()

    # Remove common answer prefixes
    answer = re.sub(
        r'^(answer\s*[:\-]?\s*)',
        '',
        answer,
        flags=re.IGNORECASE
    )

    # Remove multiple-choice letter prefix:
    # A. 34
    # B) 1/8
    # C: $7.85
    answer = re.sub(
        r'^[A-Ea-e][\.\)\:\-]\s*',
        '',
        answer
    )

    # Remove commas from numbers
    answer = answer.replace(",", "")

    # Normalize whitespace
    answer = re.sub(r'\s+', ' ', answer).strip()

    return answer


eval_df["normalized_expected"] = (
    eval_df["expected_answer"]
    .apply(normalize_answer)
)

eval_df["normalized_extracted"] = (
    eval_df["extracted_answer"]
    .apply(normalize_answer)
)

display(
    eval_df[
        [
            "expected_answer",
            "extracted_answer",
            "normalized_expected",
            "normalized_extracted"
        ]
    ].head(10)
)

,expected_answer,extracted_answer,normalized_expected,normalized_extracted
0,34,34,34,34
1,228 cm2,228 cm2,228 cm2,228 cm2
2,43,47,43,47
3,20,20,20,20
4,6,6,6,6
5,B)1/8,C)1/9,1/8,1/9
6,1/7,1/7,1/7,1/7
7,4800,"4,800",4800,4800
8,10years,10years,10years,10years
9,65,65,65,65


In [67]:
eval_df["answer_correct"] = (
    eval_df["normalized_expected"]
    == eval_df["normalized_extracted"]
)

print(
    "Correct:",
    eval_df["answer_correct"].sum()
)

print(
    "Incorrect:",
    (~eval_df["answer_correct"]).sum()
)

print(
    "Accuracy:",
    round(
        eval_df["answer_correct"].mean() * 100,
        2
    ),
    "%"
)

Correct: 1703
Incorrect: 1297
Accuracy: 56.77 %


In [ ]:
print("Kernel is responding")

In [30]:
max_new_tokens=256
repetition_penalty=1.1
do_sample=False

In [68]:
#1298 mismatch finding 
mismatches = eval_df[
    eval_df["answer_correct"] == False
].copy()

print("Total mismatches:", len(mismatches))

display(
    mismatches[
        [
            "index",
            "expected_answer",
            "extracted_answer",
            "normalized_expected",
            "normalized_extracted"
        ]
    ].head(30)
)

Total mismatches: 1297


,index,expected_answer,extracted_answer,normalized_expected,normalized_extracted
2,27604,43,47,43,47
5,3928,B)1/8,C)1/9,1/8,1/9
10,14364,C)Rs.244.83,D)Rs.245.83,Rs.244.83,Rs.245.83
11,22187,$60,$96,$60,$96
13,2885,4Rs,6Rs,4Rs,6Rs
17,16572,40,44.44,40,44.44
18,13174,33% less than N,67% less than N,33% less than N,67% less than N
20,9538,6,None,6,None
24,4519,3,4,3,4
28,7303,21,28,21,28


In [69]:
# Look at one multiple-choice question in full

mc_sample = eval_df[
    eval_df["expected_answer"].astype(str).str.contains(r"[A-E]", regex=True)
].iloc[0]

print("QUESTION:")
print(mc_sample["input"])

print("\nEXPECTED ANSWER:")
print(mc_sample["expected_answer"])

print("\nMODEL EXTRACTED ANSWER:")
print(mc_sample["extracted_answer"])

QUESTION:
Jake remembers only the last four letters of his five-letter Klingon name. If he is sure that the first letter is neither "N" nor "Z", and assuming that there are only 10 letters in the Klingon alphabet, what is the probability that he will give the correct name when asked for it by the space attendant?

Choices:
A. A)8/100
B. B)1/8
C. C)1/9
D. D)4/5
E. E)9/10

EXPECTED ANSWER:
B)1/8

MODEL EXTRACTED ANSWER:
C)1/9


In [70]:
missing_answer_df = eval_df[
    eval_df["extracted_answer"].isna()
].copy()

wrong_extracted_df = eval_df[
    eval_df["extracted_answer"].notna() &
    (~eval_df["answer_correct"])
].copy()

print("Missing Final Answer:", len(missing_answer_df))
print("Answer extracted but mismatch:", len(wrong_extracted_df))

Missing Final Answer: 280
Answer extracted but mismatch: 1017


In [71]:
import re

def extract_choice(text):
    if pd.isna(text):
        return None

    text = str(text).strip()

    # Match A, B, C, D, or E at the beginning
    match = re.match(r'^([A-Ea-e])(?:[\.\)\:\-\s]|$)', text)

    if match:
        return match.group(1).upper()

    return None


eval_df["expected_choice"] = (
    eval_df["expected_answer"]
    .apply(extract_choice)
)

eval_df["model_choice"] = (
    eval_df["extracted_answer"]
    .apply(extract_choice)
)

print(
    "Expected answers with identifiable choices:",
    eval_df["expected_choice"].notna().sum()
)

print(
    "Model answers with identifiable choices:",
    eval_df["model_choice"].notna().sum()
)

Expected answers with identifiable choices: 53
Model answers with identifiable choices: 43


In [72]:
# Identify multiple-choice questions from the question text itself

def has_choices(row):
    text = str(row["instruction"]) + "\n" + str(row["input"])
    return "Choices:" in text or "Choices" in text

eval_df["is_multiple_choice"] = eval_df.apply(has_choices, axis=1)

print("Multiple-choice questions:",
      eval_df["is_multiple_choice"].sum())

print("Non-multiple-choice questions:",
      (~eval_df["is_multiple_choice"]).sum())

Multiple-choice questions: 2098
Non-multiple-choice questions: 902


In [73]:
# Show a few multiple-choice examples

mc_df = eval_df[eval_df["is_multiple_choice"]].copy()

display(
    mc_df[
        [
            "index",
            "expected_answer",
            "extracted_answer",
            "instruction",
            "input"
        ]
    ].head(10)
)

,index,expected_answer,extracted_answer,instruction,input
1,3160,228 cm2,228 cm2,Solve the following multiple-choice math reaso...,Find the area of trapezium whose parallel side...
2,27604,43,47,Solve the following multiple-choice math reaso...,How many positive three-digit integers are div...
5,3928,B)1/8,C)1/9,Solve the following multiple-choice math reaso...,Jake remembers only the last four letters of h...
6,29031,1/7,1/7,Solve the following multiple-choice math reaso...,"On a partly cloudy day, Derek decides to walk ..."
8,4156,10years,10years,Solve the following multiple-choice math reaso...,The average age of 15 students of a class is 1...
10,14364,C)Rs.244.83,D)Rs.245.83,Solve the following multiple-choice math reaso...,Find out the C.I on Rs.4000 at 4% p.a. compoun...
11,22187,$60,$96,Solve the following multiple-choice math reaso...,John spent a total of $135 on baseball tickets...
13,2885,4Rs,6Rs,Solve the following multiple-choice math reaso...,5 men are equal to as many women as are equal ...
14,26239,5/16,5/16,Solve the following multiple-choice math reaso...,If a number N is chosen at random from the set...
16,18936,30,30,Solve the following multiple-choice math reaso...,15 men take 21 days of 8 hours each to do a pi...


In [74]:
import re
import pandas as pd
import numpy as np

eval_df = results_df.copy()

# ---------------------------------------------------------
# 1. Normalize text
# ---------------------------------------------------------

def normalize_text(x):
    if pd.isna(x):
        return None

    x = str(x).strip().lower()

    # Remove common formatting
    x = x.replace("\\", "")
    x = x.replace("**", "")
    x = x.replace("*", "")
    x = x.replace("`", "")

    # Normalize whitespace
    x = re.sub(r"\s+", " ", x)

    return x.strip()


# ---------------------------------------------------------
# 2. Extract choices from the question
# ---------------------------------------------------------

def extract_choices(row):
    text = str(row.get("input", ""))

    choices = {}

    # Match lines such as:
    # A. 335 cm2
    # B) 228 cm2
    # C. $7.85
    pattern = r"(?:^|\n)\s*([A-E])[\.\)]\s*(.+?)(?=\n\s*[A-E][\.\)]|\Z)"

    matches = re.findall(pattern, text, flags=re.IGNORECASE | re.DOTALL)

    for letter, value in matches:
        choices[letter.upper()] = value.strip()

    return choices


eval_df["choices"] = eval_df.apply(extract_choices, axis=1)

print("Rows with detected choices:",
      eval_df["choices"].apply(lambda x: len(x) > 0).sum())

Rows with detected choices: 2094


In [83]:
def extract_answer_value(answer):
    if pd.isna(answer):
        return None

    answer = str(answer).strip()

    # Remove leading choice letter:
    # B)1/8 -> 1/8
    # C)Rs.244.83 -> Rs.244.83
    answer = re.sub(
        r"^\s*[A-E]\s*[\.\)]\s*",
        "",
        answer,
        flags=re.IGNORECASE
    )

    return normalize_text(answer)


eval_df["normalized_expected"] = (
    eval_df["expected_answer"]
    .apply(extract_answer_value)
)

eval_df[
    ["expected_answer", "normalized_expected"]
].head(10)

,expected_answer,normalized_expected
0,34,34
1,228 cm2,228 cm2
2,43,43
3,20,20
4,6,6
5,B)1/8,1/8
6,1/7,1/7
7,4800,4800
8,10years,10years
9,65,65


In [80]:
import re
import pandas as pd
import numpy as np

# Work on a copy so the original results_df is untouched
eval_df = results_df.copy()

def extract_final_answer(response):
    if pd.isna(response):
        return None

    text = str(response)

    # Look for "Final Answer:" and capture what follows
    match = re.search(
        r"Final\s*Answer\s*:\s*(.+?)(?:\n|$)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        answer = match.group(1).strip()

        # Remove markdown formatting
        answer = answer.replace("**", "")
        answer = answer.replace("*", "")
        answer = answer.strip()

        return answer

    return None


eval_df["extracted_answer"] = (
    eval_df["model_response"]
    .apply(extract_final_answer)
)

print(
    "Existing extracted answers:",
    eval_df["extracted_answer"].notna().sum()
)

print(
    "Missing extracted answers:",
    eval_df["extracted_answer"].isna().sum()
)

Existing extracted answers: 2720
Missing extracted answers: 280


In [81]:
print(
    "Existing extracted answers:",
    eval_df["extracted_answer"].notna().sum()
)

print(
    "Missing extracted answers:",
    eval_df["extracted_answer"].isna().sum()
)

Existing extracted answers: 2720
Missing extracted answers: 280


In [84]:
eval_df["normalized_extracted"] = (
    eval_df["extracted_answer"]
    .apply(extract_answer_value)
)

eval_df["answer_correct"] = (
    eval_df["normalized_expected"]
    == eval_df["normalized_extracted"]
)

print("Correct:",
      eval_df["answer_correct"].sum())

print("Incorrect:",
      (~eval_df["answer_correct"]).sum())

Correct: 1699
Incorrect: 1301


In [87]:
def normalize_choice_answer(answer):
    if pd.isna(answer):
        return None

    answer = str(answer).strip()

    # If answer is ONLY A/B/C/D/E
    match = re.fullmatch(
        r"\s*([A-E])\s*[\.\)]?\s*",
        answer,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).upper()

    return None

In [88]:
eval_df["expected_choice"] = (
    eval_df["expected_answer"]
    .apply(normalize_choice_answer)
)

eval_df["model_choice"] = (
    eval_df["extracted_answer"]
    .apply(normalize_choice_answer)
)

print(
    "Expected answers that are explicit choices:",
    eval_df["expected_choice"].notna().sum()
)

print(
    "Model answers that are explicit choices:",
    eval_df["model_choice"].notna().sum()
)

Expected answers that are explicit choices: 1
Model answers that are explicit choices: 0


In [89]:
def get_choice_mapping(question):
    """
    Extract choices from the question text.
    Example:
    A. 335 cm2
    B. 228 cm2
    C. 285 cm2
    """

    if pd.isna(question):
        return {}

    text = str(question)

    pattern = r'(?m)^\s*([A-E])[\.\)]\s*(.+?)(?=\n\s*[A-E][\.\)]|\Z)'

    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    return {
        letter.upper(): normalize_text(value)
        for letter, value in matches
    }


eval_df["choice_mapping"] = eval_df["input"].apply(get_choice_mapping)

print(
    "Questions with detected choices:",
    (eval_df["choice_mapping"].apply(len) >= 2).sum()
)

print(
    "Questions without detected choices:",
    (eval_df["choice_mapping"].apply(len) < 2).sum()
)

Questions with detected choices: 2094
Questions without detected choices: 906


In [ ]:
regenerated_df.to_csv(
    retry_path,
    index=False
)

print(
    "Total regenerated:",
    len(regenerated_df)
)

In [91]:
evaluated_df = eval_df[
    eval_df["extracted_answer"].notna()
].copy()

evaluated_df["answer_correct"] = (
    evaluated_df["normalized_expected"]
    ==
    evaluated_df["normalized_extracted"]
)

correct = evaluated_df["answer_correct"].sum()
incorrect = (~evaluated_df["answer_correct"]).sum()

print("========== EVALUATION ==========")
print("Total responses:", len(eval_df))
print("Evaluated responses:", len(evaluated_df))
print("Missing Final Answer:", len(eval_df) - len(evaluated_df))
print("Correct:", correct)
print("Incorrect:", incorrect)

print(
    "Accuracy among extracted answers:",
    round(correct / len(evaluated_df) * 100, 2),
    "%"
)

========== EVALUATION ==========
Total responses: 3000
Evaluated responses: 2720
Missing Final Answer: 280
Correct: 1699
Incorrect: 1021
Accuracy among extracted answers: 62.46 %


In [ ]:
display(
    regenerated_df[
        [
            "index",
            "new_response",
            "generation_time"
        ]
    ].tail(20)
)

In [ ]:
final_df = working_df.copy()

for _, retry in regenerated_df.iterrows():

    idx = int(retry["index"])

    mask = final_df["index"].astype(int) == idx

    final_df.loc[
        mask,
        "model_response"
    ] = retry["new_response"]

    final_df.loc[
        mask,
        "generation_time"
    ] = retry["generation_time"]

In [ ]:
print("Original rows:", len(working_df))
print("Final rows:", len(final_df))

print(
    "Original unique indices:",
    working_df["index"].nunique()
)

print(
    "Final unique indices:",
    final_df["index"].nunique()
)